In [1]:
from langchain_community.vectorstores import FAISS
# from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document

d:\Assigment\GENAI_BOT\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

HF_TOKEN = os.getenv("API_KEY")

In [3]:
pip install -qU langchain-openai

Note: you may need to restart the kernel to use updated packages.


d:\Assigment\GENAI_BOT\.venv\Scripts\python.exe: No module named pip


In [4]:
pip install ipywidgets

Note: you may need to restart the kernel to use updated packages.


d:\Assigment\GENAI_BOT\.venv\Scripts\python.exe: No module named pip


In [5]:
from huggingface_hub import InferenceClient

client = InferenceClient(token=HF_TOKEN )

embedding = client.feature_extraction(
    "This is a test sentence",
    model="sentence-transformers/all-MiniLM-L6-v2"
)

print(embedding[0])

d:\Assigment\GENAI_BOT\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


0.0715524


In [7]:
from huggingface_hub import InferenceClient
import numpy as np

client = InferenceClient(token="HF_TOKEN")

documents = [
    "AI is transforming industries",
    "Machine learning is part of AI",
    "Python is used in data science"
]

# Step 1: Create embeddings
def embed(text):
    emb = client.feature_extraction(
        text,
        model="sentence-transformers/all-MiniLM-L6-v2"
    )
    return np.mean(emb, axis=0)

vectors = [embed(doc) for doc in documents]

# Step 2: Query
query = "What is machine learning?"
query_vec = embed(query)

# Step 3: Similarity (cosine)
def cosine(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

scores = [cosine(query_vec, v) for v in vectors]

# Step 4: Get best match
best_doc = documents[np.argmax(scores)]

print(best_doc)

HfHubHTTPError: Client error '401 Unauthorized' for url 'https://router.huggingface.co/hf-inference/models/sentence-transformers/all-MiniLM-L6-v2/pipeline/feature-extraction' (Request ID: Root=1-69bd98d1-2306fa1f616ba528719ac4a9;7495a94b-5a41-4fa7-8d76-fac636054c5a)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401

Invalid username or password.

In [2]:
import os
import json
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from huggingface_hub import InferenceClient

from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

# PDF support
from PyPDF2 import PdfReader

# ✅ FIX: Added docx support
from docx import Document as DocxDocument  

load_dotenv()
HF_TOKEN = os.getenv("API_KEY")


# -------------------------------
# 🔥 Custom Embedding Class
# -------------------------------
class HFEmbedding:
    def __init__(self, token):
        self.client = InferenceClient(token=token)
        self.model = "sentence-transformers/all-MiniLM-L6-v2"

    def embed_query(self, text):
        try:
            emb = self.client.feature_extraction(text, model=self.model)

            # ✅ FIX: Handle empty or invalid response
            if not emb or not isinstance(emb, list):
                return [0.0] * 384  

            # ✅ FIX: Handle nested structure safely
            vector = np.mean(emb, axis=0)

            # ✅ FIX: Ensure vector is valid
            if vector is None or len(vector) == 0:
                return [0.0] * 384

            return list(vector)

        except Exception as e:
            print(f"❌ Embedding error: {e}")
            return [0.0] * 384  # fallback

    def embed_documents(self, texts):
        # ✅ FIX: Filter empty texts
        return [self.embed_query(t) for t in texts if t.strip()]


# -------------------------------
# ✂️ Chunk Function
# -------------------------------
def chunk_text(text, size=300):
    return [text[i:i+size] for i in range(0, len(text), size) if text[i:i+size].strip()]  # ✅ FIX


# -------------------------------
# 📂 Load Documents (All Types)
# -------------------------------
def load_docs(folder="Data"):
    docs = []

    # ✅ FIX: Check folder exists
    if not os.path.exists(folder):
        raise ValueError(f"❌ Folder '{folder}' not found")

    for file in os.listdir(folder):
        file_path = os.path.join(folder, file)

        try:
            # TXT
            if file.endswith(".txt"):
                with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
                    docs.append((file, f.read()))

            # ✅ FIX: Proper DOCX handling (NOT open())
            elif file.endswith((".docx", ".doc")):
                doc = DocxDocument(file_path)
                text = "\n".join([para.text for para in doc.paragraphs])
                docs.append((file, text))

            # PDF
            elif file.endswith(".pdf"):
                reader = PdfReader(file_path)
                text = ""
                for page in reader.pages:
                    text += page.extract_text() or ""
                docs.append((file, text))

            # CSV
            elif file.endswith(".csv"):
                df = pd.read_csv(file_path)
                docs.append((file, df.to_string()))

            # JSON
            elif file.endswith(".json"):
                with open(file_path, "r", encoding="utf-8") as f:
                    data = json.load(f)
                    docs.append((file, json.dumps(data)))

        except Exception as e:
            print(f"❌ Error loading {file}: {e}")

    return docs


# -------------------------------
# 📄 Create Documents
# -------------------------------
def create_documents():
    raw_docs = load_docs()
    documents = []

    for filename, doc in raw_docs:

        # ✅ FIX: Skip empty docs
        if not doc or not doc.strip():
            continue

        chunks = chunk_text(doc, size=300)

        for chunk in chunks:
            documents.append(
                Document(
                    page_content=chunk,
                    metadata={"source": filename}
                )
            )

    return documents


# -------------------------------
# 🧠 Create FAISS Index
# -------------------------------
def create_index():
    documents = create_documents()

    # ✅ FIX: Check before FAISS
    if len(documents) == 0:
        raise ValueError("❌ No documents found. Check your Data folder.")

    print(f"✅ Total chunks created: {len(documents)}")  # DEBUG

    embeddings = HFEmbedding(token=HF_TOKEN)

    db = FAISS.from_documents(documents, embeddings)

    db.save_local("db")

    print("✅ Index Created Successfully (HuggingFace Embeddings)")


# -------------------------------
# 🚀 Run
# -------------------------------
if __name__ == "__main__":
    create_index()

d:\Assigment\GENAI_BOT\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\Assigment\GENAI_BOT\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


✅ Total chunks created: 16
❌ Embedding error: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()
❌ Embedding error: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()
❌ Embedding error: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()
❌ Embedding error: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()
❌ Embedding error: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()
❌ Embedding error: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()
❌ Embedding error: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()
❌ Embedding error: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()
❌ Embedding error: The truth value of an array with more than one element is ambiguou

`embedding_function` is expected to be an Embeddings object, support for passing in a function will soon be removed.


✅ Index Created Successfully (HuggingFace Embeddings)
